# Phase 01 — Confirm the Exact PaddleOCR Setup on Kaggle

**Project:** Bangladesh NID OCR fine-tuning  
**Environment:** Kaggle only (no local GPU / local training)  
**Docs:** `Docs/SCOPE.md`, `Docs/plan/PaddleOCR_NID_Phase_Plan.md`, `Docs/VERSION_NOTES.md`, `Docs/report/`

### What this phase does

Remove setup uncertainty before any NID training. Record exact software versions, starting models, config paths, Git pin, train/eval/export entrypoints, and how a custom Bangla + English dictionary is wired.

### In scope (model fine-tuning only)

- Starting models: `PP-OCRv5_server_det` + `PP-OCRv5_server_rec`
- One **unified Bangla + English** recognition path (no second Bengali-only OCR model)
- Later phases will fine-tune for NID layout familiarity, multi-line regions, and glyph/spacing issues

### Out of scope (do not do here)

- Image preprocessing redesign, parser/regex/field-extraction logic
- Annotating NIDs, building the final dictionary, or claiming accuracy gains
- Real NID images in Git (private Kaggle Dataset only)

### Output

Filled `VERSION_NOTES.md` under `/kaggle/working/` — copy into repo as `Docs/VERSION_NOTES.md` after review.

**Deliverable status:** D1 and D4 start. D2/D3 are later phases.

## 0. Kaggle path conventions

From the phase plan — keep these paths consistent across notebooks:

```text
/kaggle/input/<private-dataset-name>/     # read-only NID data (Phase 02+)
/kaggle/working/                          # writable experiment area
/kaggle/working/PaddleOCR/                # cloned / pinned PaddleOCR source
/kaggle/working/output/                   # checkpoints / exports / metrics (later)
```

**Hard rules:** all compute on Kaggle; attach the private dataset when needed; never commit real NID images; persist useful `/kaggle/working/` outputs as Kaggle Dataset versions when needed.

Enable a **GPU accelerator** in the Kaggle Notebook settings if available (needed for later training phases; Phase 01 still runs usefully on CPU).

## 1. Record the Kaggle environment

Capture Python, platform, accelerator / CUDA info, and writable paths — not assumptions.

In [ ]:
import sys
import os
import platform
import subprocess
from pathlib import Path

WORKDIR = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = WORKDIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PYTHON_VERSION = sys.version.split()[0]
PLATFORM = platform.platform()

print('Python:', sys.version)
print('Platform:', PLATFORM)
print('WORKDIR:', WORKDIR, '-> exists:', WORKDIR.exists())
print('INPUT_ROOT:', INPUT_ROOT, '-> exists:', INPUT_ROOT.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)

print('\n=== Accelerator / CUDA ===')
CUDA_VISIBLE = os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)')
print('CUDA_VISIBLE_DEVICES:', CUDA_VISIBLE)

nvidia_ok = False
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=30)
    nvidia_ok = result.returncode == 0
    print(result.stdout if nvidia_ok else result.stderr or 'nvidia-smi failed')
except FileNotFoundError:
    print('nvidia-smi not found (CPU session or no NVIDIA driver).')
except Exception as exc:
    print('nvidia-smi error:', exc)

ACCELERATOR = 'GPU (nvidia-smi OK)' if nvidia_ok else 'CPU / no usable GPU detected'
print('Accelerator summary:', ACCELERATOR)

## 2. Check installed Paddle packages

Reference stack from the existing NID service report (`Docs/report/OCR_Model_Report.md`):

| Package | Reference version |
| --- | --- |
| PaddleOCR | 3.3.0 |
| PaddlePaddle | 3.2.0 |
| PaddleX | 3.3.3 |

Inspect what **this** Kaggle session actually has. Pin only after confirming a compatible wheel for Kaggle's CUDA/Python.

In [ ]:
import importlib.metadata as md

REFERENCE_VERSIONS = {
    'paddleocr': '3.3.0',
    'paddlepaddle': '3.2.0',
    'paddlex': '3.3.3',
}

packages = ['paddleocr', 'paddlepaddle', 'paddlex']
versions = {}

for pkg in packages:
    try:
        versions[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        versions[pkg] = 'NOT INSTALLED'

print('Installed vs reference (from OCR_Model_Report.md):')
for pkg in packages:
    print(f'  {pkg:12} installed={versions[pkg]:<16} reference={REFERENCE_VERSIONS[pkg]}')

versions

### Optional install / pinning

Use only if packages are missing or you intentionally want the report's package family. After install, **restart the Kaggle session** and rerun from the top.

> Leave commented until you choose exact pins. Do **not** blindly install an old CUDA wheel — match Kaggle's Python/CUDA first. Then copy verified pins into repo `requirements.txt`.

In [ ]:
# Reference family from Docs/report/OCR_Model_Report.md:
# !pip install -q paddleocr==3.3.0 paddlex==3.3.3
#
# Install the PaddlePaddle build that matches this Kaggle CUDA/Python.
# Check https://www.paddlepaddle.org.cn/install/quick before running.
# Example shape only (replace with the verified wheel command):
# !pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
#
# After install: Restart session -> rerun notebook from cell 1.

## 3. Clone / pin PaddleOCR under `/kaggle/working/`

Do **not** depend on an unspecified moving `main`. Record the exact Git commit (and optional release tag). Prefer a known release tag when one matches the installed package family.

In [ ]:
PADDLEOCR_DIR = WORKDIR / 'PaddleOCR'

# Optional: set a release tag to pin (e.g. 'v2.10.0' or a tag matching your paddleocr package).
# Leave empty to use the shallow default tip, then still record the commit hash below.
PADDLEOCR_GIT_REF = os.environ.get('PADDLEOCR_GIT_REF', '').strip()

if not PADDLEOCR_DIR.exists():
    clone_cmd = [
        'git', 'clone',
        'https://github.com/PaddlePaddle/PaddleOCR.git',
        str(PADDLEOCR_DIR),
    ]
    if PADDLEOCR_GIT_REF:
        clone_cmd[2:2] = ['--branch', PADDLEOCR_GIT_REF, '--depth', '1']
    else:
        clone_cmd[2:2] = ['--depth', '1']
    subprocess.run(clone_cmd, check=True)
elif PADDLEOCR_GIT_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'tag', PADDLEOCR_GIT_REF],
                   cwd=PADDLEOCR_DIR, check=False)
    subprocess.run(['git', 'checkout', PADDLEOCR_GIT_REF], cwd=PADDLEOCR_DIR, check=False)

os.chdir(PADDLEOCR_DIR)

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
try:
    describe = subprocess.check_output(
        ['git', 'describe', '--tags', '--always'], text=True, stderr=subprocess.DEVNULL
    ).strip()
except subprocess.CalledProcessError:
    describe = '(no tag)'

print('PaddleOCR repo :', PADDLEOCR_DIR)
print('Requested ref  :', PADDLEOCR_GIT_REF or '(default shallow tip)')
print('Git commit     :', commit)
print('Describe/tag   :', describe)

## 4. Confirm the two starting models

Per `Docs/SCOPE.md` and the phase plan — these are the only starting checkpoints:

- Detection: `PP-OCRv5_server_det`
- Recognition: `PP-OCRv5_server_rec`

Do **not** switch to mobile variants. The model report notes past docs/tests that incorrectly referenced mobile models; server models are authoritative.

In [ ]:
DET_MODEL = 'PP-OCRv5_server_det'
REC_MODEL = 'PP-OCRv5_server_rec'

print('Detection model :', DET_MODEL)
print('Recognition model:', REC_MODEL)

# Where PaddleX may cache official weights (path varies by install).
candidate_model_roots = [
    Path.home() / '.paddlex' / 'official_models',
    Path('/home/appadmin/.paddlex/official_models'),
    Path('/root/.paddlex/official_models'),
    WORKDIR / '.paddlex' / 'official_models',
]

print('\nLooking for cached official model dirs (may be empty until first download):')
found_model_dirs = {}
for root in candidate_model_roots:
    if not root.exists():
        continue
    for name in (DET_MODEL, REC_MODEL):
        p = root / name
        status = 'FOUND' if p.exists() else 'missing'
        print(f'  {status}: {p}')
        if p.exists():
            found_model_dirs[name] = str(p)

if not found_model_dirs:
    print('  (none cached yet — Phase 02 inference / first train will download weights)')

## 5. Find the exact PP-OCRv5 training configs

Do not assume paths from an older tutorial.

PaddleOCR **3.x** often routes training through **PaddleX** module configs (`paddlex/configs/modules/...`), while classic `PaddleOCR/configs/...` + `tools/train.py` may still exist. Search **both** surfaces in this session and record what is actually present.

In [ ]:
def find_configs(roots, patterns):
    hits = []
    for root in roots:
        if root is None or not Path(root).exists():
            continue
        root = Path(root)
        for pat in patterns:
            hits.extend(root.rglob(pat))
    # de-dupe while preserving order
    seen, out = set(), []
    for p in hits:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out

# Classic PaddleOCR repo configs
repo_config_root = PADDLEOCR_DIR / 'configs'

# Installed PaddleX package configs (Model Report §7.4)
paddlex_roots = []
try:
    import paddlex
    paddlex_dir = Path(paddlex.__file__).resolve().parent
    paddlex_roots.append(paddlex_dir)
    print('paddlex package dir:', paddlex_dir)
except Exception as exc:
    print('paddlex not importable yet:', exc)

rec_patterns = [
    '*PP-OCRv5*server*rec*.yml',
    '*PP-OCRv5*server*rec*.yaml',
    '**/text_recognition/**/PP-OCRv5_server_rec.yaml',
    '**/text_recognition/**/PP-OCRv5_server_rec.yml',
]
det_patterns = [
    '*PP-OCRv5*server*det*.yml',
    '*PP-OCRv5*server*det*.yaml',
    '**/text_detection/**/PP-OCRv5_server_det.yaml',
    '**/text_detection/**/PP-OCRv5_server_det.yml',
]

search_roots = [repo_config_root, *paddlex_roots]
rec_candidates = find_configs(search_roots, rec_patterns)
det_candidates = find_configs(search_roots, det_patterns)

print('\nRecognition config candidates:')
for p in rec_candidates:
    print(' -', p)

print('\nDetection config candidates:')
for p in det_candidates:
    print(' -', p)

if not rec_candidates:
    print('\nWARNING: no recognition config found — adjust search or pin a different PaddleOCR/PaddleX version.')
if not det_candidates:
    print('\nWARNING: no detection config found — adjust search or pin a different PaddleOCR/PaddleX version.')

## 6. Inspect the recognition config

Locate fields for pretrained weights, character dictionary, spaces, and data lists. Key names differ between classic PaddleOCR YAML and PaddleX module YAML — record whatever this version actually uses.

In [ ]:
if not rec_candidates:
    raise FileNotFoundError(
        'No PP-OCRv5 server recognition config found. '
        'Install/pin paddleocr+paddlex and re-run section 5.'
    )

REC_CONFIG = rec_candidates[0]
print('Using recognition config:', REC_CONFIG)

rec_text = REC_CONFIG.read_text(encoding='utf-8')

rec_keywords = [
    'pretrained_model',
    'pretrain_weight_path',
    'character_dict_path',
    'use_space_char',
    'save_model_dir',
    'label_file_list',
    'data_dir',
    'dataset_dir',
    'Global.mode',
    'mode:',
]

print('\nRelevant lines:')
for i, line in enumerate(rec_text.splitlines(), start=1):
    if any(k in line for k in rec_keywords):
        print(f'{i:4}: {line}')

has_char_dict = any(k in rec_text for k in ('character_dict_path', 'char_dict_path', 'dict_path'))
has_space_flag = 'use_space_char' in rec_text
has_pretrain = any(k in rec_text for k in ('pretrained_model', 'pretrain_weight_path'))

print('\ncharacter/dict field present:', has_char_dict)
print('use_space_char present       :', has_space_flag)
print('pretrained weight field      :', has_pretrain)

## 7. Inspect the detection config

Phase 01 only needs the exact config path and training mechanics. Real NID polygon annotations come in Phase 03.

In [ ]:
if not det_candidates:
    raise FileNotFoundError(
        'No PP-OCRv5 server detection config found. '
        'Install/pin paddleocr+paddlex and re-run section 5.'
    )

DET_CONFIG = det_candidates[0]
print('Using detection config:', DET_CONFIG)

det_text = DET_CONFIG.read_text(encoding='utf-8')
det_keywords = [
    'pretrained_model',
    'pretrain_weight_path',
    'save_model_dir',
    'label_file_list',
    'data_dir',
    'dataset_dir',
]

print('\nRelevant lines:')
for i, line in enumerate(det_text.splitlines(), start=1):
    if any(k in line for k in det_keywords):
        print(f'{i:4}: {line}')

## 8. Verify train / evaluate / export entrypoints

Confirm what **this** checked-out / installed stack exposes. Do not copy an old local command blindly.

Per `OCR_Model_Report.md` §7.4, PaddleOCR 3.x often uses PaddleX modes:
`check_dataset` → `train` → `evaluate` → `export` via `python main.py -c <config> -o Global.mode=...`

Classic PaddleOCR may still expose `tools/train.py`, `tools/eval.py`, `tools/export_model.py`.

In [ ]:
classic_entrypoints = {
    'train': PADDLEOCR_DIR / 'tools' / 'train.py',
    'eval': PADDLEOCR_DIR / 'tools' / 'eval.py',
    'export': PADDLEOCR_DIR / 'tools' / 'export_model.py',
}

print('Classic PaddleOCR tools:')
for name, ep_path in classic_entrypoints.items():
    try:
        shown = ep_path.relative_to(PADDLEOCR_DIR)
    except ValueError:
        shown = ep_path
    print(f'  {name:7}: {shown} ->', 'FOUND' if ep_path.exists() else 'MISSING')

print('\nPaddleX / main.py candidates:')
paddlex_main_candidates = []
explicit = [
    PADDLEOCR_DIR / 'main.py',
]
for root in paddlex_roots:
    root = Path(root)
    explicit.append(root / 'main.py')
    explicit.append(root.parent / 'main.py')

for p in explicit:
    if p.exists():
        paddlex_main_candidates.append(p.resolve())

for root in [PADDLEOCR_DIR, *paddlex_roots]:
    root = Path(root)
    if not root.exists():
        continue
    for p in list(root.glob('main.py')) + list(root.glob('*/main.py')) + list(root.glob('*/*/main.py')):
        paddlex_main_candidates.append(p.resolve())

seen, paddlex_mains = set(), []
for p in paddlex_main_candidates:
    if p not in seen:
        seen.add(p)
        paddlex_mains.append(p)

if paddlex_mains:
    for p in paddlex_mains[:10]:
        print('  ', p)
else:
    print('  no obvious main.py under PaddleOCR/paddlex roots')

print('\nCLI module checks:')
for mod in ('paddleocr', 'paddlex'):
    try:
        r = subprocess.run(
            [sys.executable, '-m', mod, '-h'],
            capture_output=True, text=True, timeout=60,
        )
        print(f'  python -m {mod}: exit={r.returncode}')
        head = (r.stdout or r.stderr or '').strip().splitlines()[:5]
        for line in head:
            print('   ', line)
    except Exception as exc:
        print(f'  python -m {mod}: error {exc}')

TRAIN_ENTRY_CLASSIC = classic_entrypoints['train'].exists()
EVAL_ENTRY_CLASSIC = classic_entrypoints['eval'].exists()
EXPORT_ENTRY_CLASSIC = classic_entrypoints['export'].exists()
PADDLEX_MAIN = paddlex_mains[0] if paddlex_mains else None
print('\nPrimary classic train available:', TRAIN_ENTRY_CLASSIC)
print('PaddleX-style main.py available :', bool(PADDLEX_MAIN), PADDLEX_MAIN or '')


### Command templates

Templates only — later phases copy verified configs into `configs/` and point them at the private Kaggle Dataset under `/kaggle/input/...`.

Illustrative PaddleX shape from the model report (confirm keys against the installed version before budgeting training time).

In [ ]:
def rel_or_abs(path: Path) -> str:
    try:
        return str(path.relative_to(PADDLEOCR_DIR))
    except ValueError:
        return str(path)

rec_rel = rel_or_abs(REC_CONFIG)
det_rel = rel_or_abs(DET_CONFIG)

print('=== Classic PaddleOCR tools (if present) ===')
if TRAIN_ENTRY_CLASSIC:
    print('Recognition train:')
    print(f'python tools/train.py -c {rec_rel} -o Global.pretrained_model=<REC_PRETRAINED_WEIGHTS>')
    print('\nRecognition eval:')
    print(f'python tools/eval.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT>')
    print('\nRecognition export:')
    print(f'python tools/export_model.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT> Global.save_inference_dir=<REC_EXPORT_DIR>')
    print('\nDetection train:')
    print(f'python tools/train.py -c {det_rel} -o Global.pretrained_model=<DET_PRETRAINED_WEIGHTS>')
    print('\nDetection eval:')
    print(f'python tools/eval.py -c {det_rel} -o Global.pretrained_model=<DET_CHECKPOINT>')
    print('\nDetection export:')
    print(f'python tools/export_model.py -c {det_rel} -o Global.pretrained_model=<DET_CHECKPOINT> Global.save_inference_dir=<DET_EXPORT_DIR>')
else:
    print('(classic tools/*.py not found in this checkout)')

print('\n=== PaddleX module workflow (PaddleOCR 3.x — confirm on this install) ===')
print('# Validate dataset')
print(f'python main.py -c {rec_rel} -o Global.mode=check_dataset -o Global.dataset_dir=/kaggle/input/<rec-dataset>')
print('# Train from pretrained')
print(f'python main.py -c {rec_rel} -o Global.mode=train -o Global.dataset_dir=/kaggle/input/<rec-dataset> -o Train.pretrain_weight_path=<REC_PRETRAINED_WEIGHTS>')
print('# Evaluate / export')
print(f'python main.py -c {rec_rel} -o Global.mode=evaluate')
print(f'python main.py -c {rec_rel} -o Global.mode=export')
print('\n# Same pattern for detection:')
print(f'python main.py -c {det_rel} -o Global.mode=train -o Global.dataset_dir=/kaggle/input/<det-dataset> -o Train.pretrain_weight_path=<DET_PRETRAINED_WEIGHTS>')

COMMAND_NOTES = (
    'Prefer the entrypoint that exists on this Kaggle install. '
    'Confirm option keys against installed paddlex before Phase 06/07. '
    'Always start from pretrained weights — do not train PP-OCRv5 from scratch on 2–4 NIDs.'
)
print('\nNote:', COMMAND_NOTES)

## 9. Verify custom Bangla + English dictionary mechanism

Phase 01 does **not** build the final NID dictionary (that is Phase 05).

Here we only verify that:

1. The recognition config exposes a custom dictionary / charset path, and
2. This Kaggle environment can read/write UTF-8 Bangla characters safely.

Task requirement: one unified Bangla + English vocabulary inside this same PaddleOCR recognizer — not a separate Bengali engine.

In [ ]:
print('character/dict field present:', has_char_dict)
print('use_space_char present       :', has_space_flag)

TEST_DICT = WORKDIR / 'bangla_english_test_dict.txt'

test_chars = [
    # English + digits (NID Latin fields)
    'A', 'B', 'C', 'M', 'D',
    '0', '1', '2',
    # Bangla letters / signs (fathers/mothers name, address, etc.)
    'অ', 'আ', 'ই',
    'ক', 'খ', 'গ',
    'া', 'ি', 'ী', 'ু', 'ূ', '্', 'ং', 'ঃ', 'ঁ',
    # punctuation seen on cards
    '.', '-', '/', ' ',
]

TEST_DICT.write_text('\n'.join(test_chars) + '\n', encoding='utf-8')
roundtrip = TEST_DICT.read_text(encoding='utf-8')
bangla_ok = all(ch in roundtrip for ch in ('অ', 'ক', '্', 'ঁ'))

print(roundtrip)
print('Dictionary test file:', TEST_DICT)
print('UTF-8 Bangla round-trip:', 'PASS' if bangla_ok else 'FAIL')

## 10. Generate `VERSION_NOTES.md`

Main Phase 01 output. Shape matches repo template `Docs/VERSION_NOTES.md`.

After review, copy `/kaggle/working/VERSION_NOTES.md` into the GitHub repo as `Docs/VERSION_NOTES.md` (code/docs only — no NID images).

In [ ]:
VERSION_NOTES = WORKDIR / 'VERSION_NOTES.md'

det_model_path = found_model_dirs.get(DET_MODEL, 'Not cached yet — download on first use')
rec_model_path = found_model_dirs.get(REC_MODEL, 'Not cached yet — download on first use')

classic_train_cmd = (
    f'python tools/train.py -c {rec_rel} -o Global.pretrained_model=<REC_PRETRAINED_WEIGHTS>'
    if TRAIN_ENTRY_CLASSIC else 'classic tools/train.py not found'
)
paddlex_train_cmd = (
    f'python main.py -c {rec_rel} -o Global.mode=train '
    f'-o Global.dataset_dir=/kaggle/input/<rec-dataset> '
    f'-o Train.pretrain_weight_path=<REC_PRETRAINED_WEIGHTS>'
)

notes = f'''# Environment & PaddleOCR Version Notes

> Generated by `notebooks/phase01_paddleocr_kaggle.ipynb` on Kaggle.  
> Copy into the repo as `Docs/VERSION_NOTES.md` after manual review.

## Python

Version: {PYTHON_VERSION}

Platform: {PLATFORM}

Accelerator: {ACCELERATOR}

## PaddlePaddle

Version: {versions.get('paddlepaddle')}

Reference (OCR_Model_Report.md): 3.2.0

## PaddleOCR

Version: {versions.get('paddleocr')}

Reference (OCR_Model_Report.md): 3.3.0

Git commit: `{commit}`

Describe / tag: `{describe}`

Requested ref: `{PADDLEOCR_GIT_REF or 'default shallow tip'}`

Clone path: `{PADDLEOCR_DIR}`

## PaddleX

Version: {versions.get('paddlex')}

Reference (OCR_Model_Report.md): 3.3.3

## Target Models

Detection:
{DET_MODEL}

Recognition:
{REC_MODEL}

Cached detection model path: `{det_model_path}`

Cached recognition model path: `{rec_model_path}`

## Training Entry Points

Classic tools/train.py present: {TRAIN_ENTRY_CLASSIC}

Classic tools/eval.py present: {EVAL_ENTRY_CLASSIC}

Classic tools/export_model.py present: {EXPORT_ENTRY_CLASSIC}

PaddleX-style main.py: `{PADDLEX_MAIN or 'not found'}`

Note: PaddleOCR 3.x often trains via PaddleX (`Global.mode=train|evaluate|export`). Confirm which surface this install uses before Phase 06/07.

## Detection Config

`{det_rel}`

Absolute: `{DET_CONFIG}`

## Recognition Config

`{rec_rel}`

Absolute: `{REC_CONFIG}`

## Bangla + English Dictionary Behaviour

- Unified Bangla + English path inside this same recognizer (no second Bengali OCR model).
- Config exposes character/dict path field: {has_char_dict}
- `use_space_char` present: {has_space_flag}
- UTF-8 Bangla dictionary file round-trip on Kaggle: {'PASS' if bangla_ok else 'FAIL'}
- Final NID charset is built in Phase 05 from real labels — not in Phase 01.

## Train Command

Classic (if available):

```bash
{classic_train_cmd}
```

PaddleX-style (confirm keys on install):

```bash
{paddlex_train_cmd}
```

## Evaluation Command

```bash
python tools/eval.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT>
# or
python main.py -c {rec_rel} -o Global.mode=evaluate
```

## Export Command

```bash
python tools/export_model.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT> Global.save_inference_dir=<REC_EXPORT_DIR>
# or
python main.py -c {rec_rel} -o Global.mode=export
```

## Kaggle Paths

- `/kaggle/input/<private-dataset-name>/` — NID data (Phase 02+)
- `/kaggle/working/PaddleOCR/` — pinned source
- `/kaggle/working/output/` — checkpoints / exports (later phases)

## Phase 01 Checklist

- [x] Kaggle environment inspected (Python, accelerator/CUDA)
- [x] Paddle package versions recorded
- [x] PP-OCRv5_server_det confirmed
- [x] PP-OCRv5_server_rec confirmed
- [x] PaddleOCR Git commit / tag recorded
- [x] Exact detection config located
- [x] Exact recognition config located
- [x] Train/eval/export entrypoints located (classic and/or PaddleX)
- [x] Custom character dictionary mechanism located
- [x] Bangla Unicode file handling verified

## Intentionally deferred

- Real NID baseline inference — Phase 02
- Detection annotations — Phase 03
- Recognition crops/labels — Phase 04
- Final Bangla + English dictionary — Phase 05
- Recognition fine-tuning — Phase 06
- Detection fine-tuning — Phase 07
'''

VERSION_NOTES.write_text(notes, encoding='utf-8')
print(notes)
print('\nSaved to:', VERSION_NOTES)
print('Copy this file into the repo as Docs/VERSION_NOTES.md after review.')

## Phase 01 done when

You can answer (from `Docs/plan/PaddleOCR_NID_Phase_Plan.md`):

1. Which PaddleOCR / PaddlePaddle / PaddleX versions are we using on Kaggle?
2. Which detector and recognizer are we starting from?
3. Where are their exact training configs (and which stack — classic tools vs PaddleX)?
4. Which scripts / modes perform train / evaluate / export?
5. Where does the recognition dictionary enter the config?
6. Can this Kaggle environment safely handle Bangla Unicode labels / dictionary text?

If yes, Phase 01 is complete. Copy `VERSION_NOTES.md` into `Docs/`, then move to **Phase 02: run the stock model on 2–4 NIDs and save baseline outputs** under `/kaggle/working/experiments/phase02_baseline/`.

Remember: 2–4 NIDs validate the **process**, not accuracy (`Docs/SCOPE.md`).